# Continuous Collision Detection Algorithm Analysis: T-CCD vs Swept AABB

## Overview

This notebook provides a comprehensive analysis of two collision detection algorithms:

- **T-CCD (Trajectory-based Continuous Collision Detection)**: A continuous collision detection method that uses trajectory analysis
- **Swept AABB (Time of Impact CCD)**: A swept bounding box approach for continuous collision detection

## Analysis Scope

We analyze these algorithms across multiple dimensions:

1. **Collision Accuracy Detection**: Missed Collision and True Collision Detection
2. **Performance Metrics**: Processing time and Frame per seconds (fps)
3. **Statistical Significance**: Hypothesis testing and confidence intervals

The parameters perform in order to get the simulation is seed 45, fps 60, velocity of 1500 units/sec. 



## 1. Setup and Data Loading

This section handles the initial setup and loading of collision detection data from CSV files. We load violation data for both algorithms across multiple categories:

- **Boundary Violations**: Particles crossing simulation boundaries
- **Conservation Violations**: Energy/momentum conservation errors  
- **Initial Overlaps**: Particles starting in overlapping positions
- **Missed Collisions**: Undetected actual collisions
- **Events Data**: Collision event records with timing information

The data loading function automatically aggregates multiple CSV files per algorithm and adds algorithm labels for comparative analysis.

In [65]:
import os
import glob
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from PIL import Image
from IPython.display import display 
import kaleido


# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10



In [66]:
log_path = './data/velocity1500/'

def load_simulation_data(file_type, algorithm_name, glob_pattern, algorithm_label):
    files = glob.glob(os.path.join (log_path, glob_pattern))
    dfs = []

    for f in files:
        df = pd.read_csv(f)
        df['algorithm'] = algorithm_label
        dfs.append(df)

    if not dfs:
        print(f" No files found for pattern: {glob_pattern}")
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)

# Load all datasets
tccd_conservation_violations_df = load_simulation_data('conservation', 'tccd', 'conservation_violations_tccd_500.csv', 'T-CCD')
tccd_initial_overlaps_df = load_simulation_data('initial_overlaps', 'tccd', 'initial_overlaps_tccd_500.csv', 'T-CCD')
tccd_missed_collisions_df = load_simulation_data('missed_collisions', 'tccd', 'missed_collisions_tccd_500.csv', 'T-CCD')
tccd_true_positive_df = load_simulation_data('true_positives', 'tccd', 'true_positives_tccd_500.csv', 'T-CCD')
tccd_events_df = load_simulation_data('events', 'tccd', 'events_tccd_500.csv', 'T-CCD')

swept_aabb_conservation_violations_df = load_simulation_data('conservation', 'swept_aabb', 'conservation_violations_swept_aabb_500.csv', 'Swept AABB')
swept_aabb_initial_overlaps_df = load_simulation_data('initial_overlaps', 'swept_aabb', 'initial_overlaps_swept_aabb_500.csv', 'Swept AABB')   
swept_aabb_missed_collisions_df = load_simulation_data('missed_collisions', 'swept_aabb', 'missed_collisions_swept_aabb_500.csv', 'Swept AABB')
swept_aabb_true_positive_df = load_simulation_data('true_positives', 'swept_aabb', 'true_positives_swept_aabb_500.csv', 'Swept AABB')
swept_aabb_events_df = load_simulation_data('events', 'swept_aabb', 'events_swept_aabb_500.csv', 'Swept AABB')


## 2. Data Preprocessing and Aggregation

The aggregation function `count_per_frame()` groups violations by algorithm and frame number, creating a unified performance DataFrame that combines all violation types for comprehensive analysis.

In [67]:

def count_per_frame(df, name):
    if df.empty:
        print(f"  {name}: No data found")
        return pd.DataFrame(columns=['algorithm', 'frame', name])
    
    print(f" {name}: {len(df)} total records across {df['frame'].nunique()} frames")
    
    grouped = df.groupby(['algorithm', 'frame']).size().reset_index(name=name)
    return grouped


print(" Processing violation data by frame...\n")

missed_count = count_per_frame(pd.concat([tccd_missed_collisions_df, swept_aabb_missed_collisions_df], ignore_index=True), 'missed_collisions')
cons_err_count = count_per_frame(pd.concat([tccd_conservation_violations_df, swept_aabb_conservation_violations_df], ignore_index=True), 'conservation_errors')
true_pos_count = count_per_frame(pd.concat([tccd_true_positive_df, swept_aabb_true_positive_df], ignore_index=True), 'true_positives')


performance_df = missed_count.merge(cons_err_count, on=['algorithm', 'frame'], how='outer')\
                             .merge(true_pos_count, on=['algorithm', 'frame'], how='outer') \
                             .fillna(0)

# Convert counts to integers
performance_df[['missed_collisions', 'conservation_errors']] = performance_df[['missed_collisions', 'conservation_errors']].astype(int)

print(f"\nSuccessfully created performance DataFrame with {len(performance_df)} rows")
print("\nFirst few rows:")
display(performance_df.head())

 Processing violation data by frame...

 missed_collisions: 4036 total records across 1052 frames
 conservation_errors: 2400 total records across 1200 frames
 true_positives: 165213 total records across 1200 frames

Successfully created performance DataFrame with 2402 rows

First few rows:


,algorithm,frame,missed_collisions,conservation_errors,true_positives
0,Swept AABB,1,23,0,100.0
1,Swept AABB,2,1,1,89.0
2,Swept AABB,3,1,1,100.0
3,Swept AABB,4,0,1,100.0
4,Swept AABB,5,2,1,97.0


### 2.1 Descriptive Statistics on Error Metrics

In [68]:
error_metrics = ['missed_collisions', 'conservation_errors']

algo_summary = performance_df.groupby('algorithm')[error_metrics].agg(['mean', 'std', 'min', 'max', 'sum']).round(2)
display(algo_summary)
print("\n")

missed_collisions                     conservation_errors        \
                        mean   std min max   sum                mean   std   
algorithm                                                                    
Swept AABB              1.66  1.81   0  23  1991                 1.0  0.03   
T-CCD                   1.70  2.39   0  35  2045                 1.0  0.03   

                          
           min max   sum  
algorithm                 
Swept AABB   0   1  1200  
T-CCD        0   1  1200

## 3. Exploratory Data Analysis (EDA)

This section performs comprehensive exploratory data analysis to understand the characteristics and patterns in our collision detection data. The EDA includes:

### 3.1 Collision Detection Accuracy
- Each algorithm provides the number of missed collisions per frame in order to see the accuracy and stability of the two.
- Number of undetected collision or miscalculated collisions
- The acceptable range is <1% of total collisions


#### 3.1 Collision Detection Accuracy

In [69]:
tccd_valid_collisions = performance_df[performance_df['algorithm']=='T-CCD']['true_positives'].sum()
print(f"T-CCD Valid Collisions: {tccd_valid_collisions}")
swept_aabb_valid_collisions = performance_df[performance_df['algorithm']=='Swept AABB']['true_positives'].sum()
print(f"Swept AABB Valid Collisions: {swept_aabb_valid_collisions}")
tccd_missed_collisions = performance_df[performance_df['algorithm']=='T-CCD']['missed_collisions'].sum()
print(f"T-CCD Missed Collisions: {tccd_missed_collisions}")
swept_aabb_missed_collisions = performance_df[performance_df['algorithm']=='Swept AABB']['missed_collisions'].sum()
print(f"Swept AABB Missed Collisions: {swept_aabb_missed_collisions}")

tccd_error_rate = tccd_missed_collisions / (tccd_valid_collisions + tccd_missed_collisions) * 100
swept_aabb_error_rate = swept_aabb_missed_collisions / (swept_aabb_valid_collisions + swept_aabb_missed_collisions) * 100
print(f"T-CCD Error Rate: {tccd_error_rate:.6f}%")
print(f"Swept AABB Error Rate: {swept_aabb_error_rate:.6f}%")

tccd_accuracy = 100 - tccd_error_rate
swept_aabb_accuracy = 100 - swept_aabb_error_rate
print(f"T-CCD Accuracy: {tccd_accuracy:.6f}%")
print(f"Swept AABB Accuracy: {swept_aabb_accuracy:.6f}%")

T-CCD Valid Collisions: 85492.0
Swept AABB Valid Collisions: 79721.0
T-CCD Missed Collisions: 2045
Swept AABB Missed Collisions: 1991
T-CCD Error Rate: 2.336155%
Swept AABB Error Rate: 2.436607%
T-CCD Accuracy: 97.663845%
Swept AABB Accuracy: 97.563393%


## 4. Collision Detection Accuracy Analysis  (Proportion Z-Test)

The Proportion Z-Test is a statistical method used to compare the success rates of two systems based on a binary outcome, including whether a collision was correctly detected or not. In this study, it will be employed to assess whether the T-CCD system statistically outperforms the Time of Impact method in terms of collision detection accuracy.

Specifically, it compares the collision detection success rate and the false negative rate (i.e., missed collisions) between the two systems. The test evaluates whether the observed difference in proportions is statistically significant or simply due to chance.

**The formula for the Z-test is given by:**

$$Z = \frac{p_1 - p_2}{\sqrt{p(1-p)\left(\frac{1}{n_1} + \frac{1}{n_2}\right)}}$$

**Where:**
- $p_1$ and $p_2$ are the sample proportions of success in group 1 (T-CCD) and group 2 (TOI-CCD), respectively
- $n_1$ and $n_2$ are the sample sizes of the two groups, $n$ = true positive + false negative
- $p$ is the pooled proportion of success rate: $p = \frac{x_1 + x_2}{n_1 + n_2}$

In [70]:
def two_proportion_z_test(tp1, fn1, tp2, fn2, alpha=0.05):
    n1 = tp1 + fn1 # True Positives + False Negatives
    n2 = tp2 + fn2 # True Positives + False Negatives
    if n1 == 0 or n2 == 0:
        raise ValueError("n1 and n2 must be > 0")
    
    p1 = tp1 / n1
    p2 = tp2 / n2
    
    # Pooled proportion
    p_pooled = (tp1 + tp2) / (n1 + n2)
    
    # Standard error
    se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n1 + 1/n2))
    if se == 0:
        raise ValueError("Pooled standard error is zero (degenerate). Check inputs.")
    
    # Z-statistic
    z_stat = (p1 - p2) / se
    
    # Two-tailed p-value
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    # Confidence interval for difference in proportions
    se_diff = np.sqrt((p1 * (1-p1)/n1) + (p2 * (1-p2)/n2))
    z_crit = stats.norm.ppf(1 - alpha/2)
    diff = p1 - p2
    ci = (diff - z_crit * se_diff, diff + z_crit * se_diff)
    
    return {
        'p1': p1,
        'p2': p2,
        'diff': diff,
        'z_stat': z_stat,
        'p_value': p_value,
        'ci_lower': ci[0],
        'ci_upper': ci[1],
        'n1': n1,
        'n2': n2
    }

print("=== HYPOTHESIS 1: ACCURACY COMPARISON ===\n")

# Use full dataset
tccd_frame_data = tccd_events_df[tccd_events_df['frame'] <= 1200]
swept_frame_data = swept_aabb_events_df[swept_aabb_events_df['frame'] <= 1200]

print(f"T-CCD total events: {len(tccd_frame_data['frame']):,}")
print(f"Swept AABB total events: {len(swept_frame_data['frame']):,}\n")
# Get collision counts
tccd_tp = tccd_valid_collisions
tccd_fn = tccd_missed_collisions
swept_tp = swept_aabb_valid_collisions
swept_fn = swept_aabb_missed_collisions

z_test = two_proportion_z_test(tccd_tp, tccd_fn, swept_tp, swept_fn)

print("Sample Sizes:")
print("-" * 50)
print(f"T-CCD total events: {z_test['n1']:,}")
print(f"Swept AABB total events: {z_test['n2']:,}\n")

print("Accuracy Analysis (True Positive Rate)")
print("-" * 50)
print(f"T-CCD TP Rate: {z_test['p1']:.4%}")
print(f"Swept AABB TP Rate: {z_test['p2']:.4%}")
print(f"Absolute Difference: {abs(z_test['diff']):.4%}")
print(f"95% CI: [{z_test['ci_lower']:.4%}, {z_test['ci_upper']:.4%}]")
print(f"Z-statistic: {z_test['z_stat']:.4f}")
print(f"P-value: {z_test['p_value']:.4e}")
print(f"Statistical Significance: {'Significant' if z_test['p_value'] < 0.05 else 'Not Significant'}")

=== HYPOTHESIS 1: ACCURACY COMPARISON ===

T-CCD total events: 85,492
Swept AABB total events: 79,721

Sample Sizes:
--------------------------------------------------
T-CCD total events: 87,537.0
Swept AABB total events: 81,712.0

Accuracy Analysis (True Positive Rate)
--------------------------------------------------
T-CCD TP Rate: 97.6638%
Swept AABB TP Rate: 97.5634%
Absolute Difference: 0.1005%
95% CI: [-0.0451%, 0.2460%]
Z-statistic: 1.3535
P-value: 1.7589e-01
Statistical Significance: Not Significant


## 4. Welch's T-Test Analysis

### Hypotheses

**Null Hypothesis (H₀):** There is no significant difference between the T-CCD simulator and the TOI-CCD method regarding simulation accuracy, computational cost, memory usage, and CPU load.

**Alternative Hypothesis (H₁):** There is a significant difference between T-CCD simulator than the TOI-CCD for simulation accuracy, computational cost, memory usage, and CPU load.

We use Welch's t-test to evaluate whether T-CCD method shows significantly different performance metrics compared to the traditional TOI-CCD (Swept AABB) method. This statistical test accounts for potentially unequal variances between the two methods and is appropriate for continuous data from independent samples.

In [71]:
def calculate_frame_durations(events_df):
    """Calculate frame durations from cumulative timestamps"""
    frame_times = events_df.groupby('frame')['time_s'].first().sort_index()
    

    frame_durations = frame_times.shift(-1) - frame_times
    
    if not frame_durations.empty:

        last_frame = frame_times.index[-1]
        last_frame_events = events_df[events_df['frame'] == last_frame]
        last_frame_end_time = last_frame_events['time_s'].max()
        last_frame_start_time = frame_times.iloc[-1]
        last_frame_duration = last_frame_end_time - last_frame_start_time
        

        frame_durations.iloc[-1] = last_frame_duration
    

    frame_durations = frame_durations.dropna()
    
    return frame_durations, frame_times


def welch_dof(data1, data2):
    """Calculate degrees of freedom using Welch–Satterthwaite equation."""
    s1, s2 = np.var(data1, ddof=1), np.var(data2, ddof=1)
    n1, n2 = len(data1), len(data2)
    numerator = (s1/n1 + s2/n2)**2
    denominator = ((s1/n1)**2 / (n1 - 1)) + ((s2/n2)**2 / (n2 - 1))
    return numerator / denominator

def welch_t_test(data1, data2, alpha=0.05, alternative='two-sided'):
    """Perform Welch's t-test with confidence interval and summary statistics."""
    t_stat, p_value = stats.ttest_ind(data1, data2, equal_var=False, alternative=alternative)

    mean1, std1 = np.mean(data1), np.std(data1, ddof=1)
    mean2, std2 = np.mean(data2), np.std(data2, ddof=1)
    n1, n2 = len(data1), len(data2)

    df = welch_dof(data1, data2)
    se = np.sqrt(std1**2/n1 + std2**2/n2)
    t_crit = stats.t.ppf(1 - alpha/2, df)
    ci = (mean1 - mean2 - t_crit * se, mean1 - mean2 + t_crit * se)

      
    # Calculate Cohen's d effect size
    pooled_std = np.sqrt(((n1 - 1) * std1**2 + (n2 - 1) * std2**2) / (n1 + n2 - 2))
    cohens_d = (mean1 - mean2) / pooled_std

    return {
        'mean1': mean1,
        'mean2': mean2,
        'std1': std1,
        'std2': std2,
        'diff': mean1 - mean2,
        't_stat': t_stat,
        'p_value': p_value,
        'df': df,
        'ci_lower': ci[0],
        'ci_upper': ci[1],
        'cohens_d': cohens_d,
        'n1': n1,
        'n2': n2
    }

print("\n=== HYPOTHESIS 2: PERFORMANCE COMPARISON ===\n")

# Calculate frame durations for full dataset
tccd_frame_durations, tccd_frame_times = calculate_frame_durations(tccd_frame_data)
swept_frame_durations, swept_frame_times = calculate_frame_durations(swept_frame_data)


# Test frame duration differences using Welch's t-test
time_test = welch_t_test(tccd_frame_durations.values, swept_frame_durations.values, 0.05, 'two-sided')

print("Sample Sizes:")
print("-" * 50)
print(f"T-CCD frames: {time_test['n1']:,}")
print(f"Swept AABB frames: {time_test['n2']:,}\n")

print("Frame Duration Analysis (Full Dataset)")
print("-" * 50)
print(f"T-CCD Mean Frame Duration ± SD: {time_test['mean1']:.6f} ± {time_test['std1']:.6f} seconds")
print(f"Swept AABB Mean Frame Duration ± SD: {time_test['mean2']:.6f} ± {time_test['std2']:.6f} seconds")
print(f"Mean Difference: {time_test['diff']:.6f} seconds")
print(f"95% CI: [{time_test['ci_lower']:.6f}, {time_test['ci_upper']:.6f}]")
print(f"T-statistic: {time_test['t_stat']:.4f}")
print(f"Degrees of Freedom: {time_test['df']:.2f}")
print(f"P-value: {time_test['p_value']:.4e}")
print(f"Cohen's d: {time_test['cohens_d']:.4f}")
print(f"Statistical Significance: {'Significant' if time_test['p_value'] < 0.05 else 'Not Significant'}")

# Interpretation
if time_test['diff'] > 0:
    faster_algo = "Swept AABB"
    slower_algo = "T-CCD"
else:
    faster_algo = "T-CCD"
    slower_algo = "Swept AABB"
    
print(f"\nInterpretation: {faster_algo} has faster frame processing on average")

# Calculate comprehensive FPS metrics using full dataset
print("\nComprehensive FPS Analysis (Full Dataset)")
print("-" * 50)

# Total simulation metrics
tccd_total_time = tccd_frame_durations.sum()
swept_total_time = swept_frame_durations.sum()
tccd_total_frames = len(tccd_frame_durations)
swept_total_frames = len(swept_frame_durations)

# Average FPS calculations
tccd_avg_fps = tccd_total_frames / tccd_total_time
swept_avg_fps = swept_total_frames / swept_total_time

# Frame-by-frame FPS (instantaneous FPS for each frame)
tccd_instantaneous_fps = 1.0 / tccd_frame_durations
swept_instantaneous_fps = 1.0 / swept_frame_durations

print(f"Simulation Overview:")
print(f"  T-CCD: {tccd_total_frames:,} frames in {tccd_total_time:.3f} seconds ({tccd_total_time/60:.2f} minutes)")
print(f"  Swept AABB: {swept_total_frames:,} frames in {swept_total_time:.3f} seconds ({swept_total_time/60:.2f} minutes)")

print(f"\nAverage Performance:")
print(f"  T-CCD Average FPS: {tccd_avg_fps:.3f} fps")
print(f"  Swept AABB Average FPS: {swept_avg_fps:.3f} fps")
print(f"  Performance Difference: {abs(tccd_avg_fps - swept_avg_fps):.3f} fps")
print(f"  Faster Algorithm: {'T-CCD' if tccd_avg_fps > swept_avg_fps else 'Swept AABB'}")
print(f"  Speed Improvement: {max(tccd_avg_fps, swept_avg_fps) / min(tccd_avg_fps, swept_avg_fps):.2f}x")

print(f"\nInstantaneous FPS Statistics:")
print(f"  T-CCD FPS: Mean={tccd_instantaneous_fps.mean():.3f}, Std={tccd_instantaneous_fps.std():.3f}")
print(f"  Swept AABB FPS: Mean={swept_instantaneous_fps.mean():.3f}, Std={swept_instantaneous_fps.std():.3f}")

# Performance relative to real-time standards
print(f"\nReal-time Performance Assessment:")
print(f"  T-CCD: {(tccd_avg_fps/60)*100:.1f}% of 60 FPS standard")
print(f"  Swept AABB: {(swept_avg_fps/60)*100:.1f}% of 60 FPS standard")





=== HYPOTHESIS 2: PERFORMANCE COMPARISON ===

Sample Sizes:
--------------------------------------------------
T-CCD frames: 1,200
Swept AABB frames: 1,200

Frame Duration Analysis (Full Dataset)
--------------------------------------------------
T-CCD Mean Frame Duration ± SD: 0.019157 ± 0.006656 seconds
Swept AABB Mean Frame Duration ± SD: 0.016861 ± 0.002052 seconds
Mean Difference: 0.002296 seconds
95% CI: [0.001902, 0.002691]
T-statistic: 11.4205
Degrees of Freedom: 1424.80
P-value: 5.7587e-29
Cohen's d: 0.4662
Statistical Significance: Significant

Interpretation: Swept AABB has faster frame processing on average

Comprehensive FPS Analysis (Full Dataset)
--------------------------------------------------
Simulation Overview:
  T-CCD: 1,200 frames in 22.989 seconds (0.38 minutes)
  Swept AABB: 1,200 frames in 20.233 seconds (0.34 minutes)

Average Performance:
  T-CCD Average FPS: 52.200 fps
  Swept AABB Average FPS: 59.309 fps
  Performance Difference: 7.109 fps
  Faster Algori

### 4.1 Conservation Error Statistical Analysis

To determine if the observed differences in conservation errors between algorithms are statistically significant, we perform formal hypothesis testing using Welch's t-tests for each error type. This provides evidence-based conclusions rather than relying solely on descriptive statistics.

In [72]:
# Statistical Testing for Conservation Errors
print("="*80)
print("CONSERVATION ERROR STATISTICAL ANALYSIS")
print("="*80)

tccd_conservation_violations = pd.read_csv(os.path.join(log_path, 'conservation_violations_tccd_500.csv'))
swept_aabb_conservation_violations = pd.read_csv(os.path.join(log_path, 'conservation_violations_swept_aabb_500.csv'))

tccd_conservation_energy_errors = tccd_conservation_violations['energy_error']
swept_aabb_conservation_energy_errors = swept_aabb_conservation_violations['energy_error']
tccd_conservation_x_errors = tccd_conservation_violations['x_error']
swept_aabb_conservation_x_errors = swept_aabb_conservation_violations['x_error']
tccd_conservation_y_errors = tccd_conservation_violations['y_error']
swept_aabb_conservation_y_errors = swept_aabb_conservation_violations['y_error']


energy_test = welch_t_test(tccd_conservation_energy_errors, swept_aabb_conservation_energy_errors,  0.05, 'two-sided')
x_error_test = welch_t_test(tccd_conservation_x_errors, swept_aabb_conservation_x_errors,  0.05, 'two-sided')
y_error_test = welch_t_test(tccd_conservation_y_errors, swept_aabb_conservation_y_errors,  0.05, 'two-sided')

# Create comprehensive results table
conservation_results_data = {
    'Error Type': ['Energy Error', 'X-axis Error', 'Y-axis Error'],
    'T-CCD Mean ± SD': [
        f"{energy_test['mean1']:.4f} ± {energy_test['std1']:.4f}",
        f"{x_error_test['mean1']:.4f} ± {x_error_test['std1']:.4f}",
        f"{y_error_test['mean1']:.4f} ± {y_error_test['std1']:.4f}"
    ],
    'Swept AABB Mean ± SD': [
        f"{energy_test['mean2']:.4f} ± {energy_test['std2']:.4f}",
        f"{x_error_test['mean2']:.4f} ± {x_error_test['std2']:.4f}",
        f"{y_error_test['mean2']:.4f} ± {y_error_test['std2']:.4f}"
    ],
    'Mean Difference': [
        f"{energy_test['diff']:.4f}",
        f"{x_error_test['diff']:.4f}",
        f"{y_error_test['diff']:.4f}"
    ],
    'P-value': [
        f"{energy_test['p_value']:.4e}",
        f"{x_error_test['p_value']:.4e}",
        f"{y_error_test['p_value']:.4e}"
    ],
    'Statistical Significance': [
        'YES' if energy_test['p_value'] < 0.05 else 'NO',
        'YES' if x_error_test['p_value'] < 0.05 else 'NO',
        'YES' if y_error_test['p_value'] < 0.05 else 'NO'
    ],
    "Cohen's d": [
        f"{energy_test['cohens_d']:.3f}",
        f"{x_error_test['cohens_d']:.3f}",
        f"{y_error_test['cohens_d']:.3f}"
    ]
}

conservation_results_df = pd.DataFrame(conservation_results_data)
print("\n**Table 3: Conservation Error Comparison Results**")
print("-" * 80)
display(conservation_results_df)

# Detailed statistical reporting
print(f"\n" + "="*60)
print("DETAILED STATISTICAL RESULTS")
print("="*60)

error_tests = [
    ('Energy Conservation Error', energy_test),
    ('X-axis Positional Error', x_error_test), 
    ('Y-axis Positional Error', y_error_test)
]

for error_name, test_result in error_tests:
    print(f"\n**{error_name}:**")
    print("-" * 50)
    print(f"  T-CCD: {test_result['mean1']:.4f} ± {test_result['std1']:.4f}")
    print(f"  Swept AABB: {test_result['mean2']:.4f} ± {test_result['std2']:.4f}")
    print(f"  Mean Difference: {test_result['diff']:.4f}")
    print(f"  95% Confidence Interval: [{test_result['ci_lower']:.4f}, {test_result['ci_upper']:.4f}]")
    print(f"  T-statistic: {test_result['t_stat']:.3f}")
    print(f"  P-value: {test_result['p_value']:.4e}")
    print(f"  Cohen's d: {test_result['cohens_d']:.3f}")
    
    # Effect size interpretation
    cohens_d = abs(test_result['cohens_d'])
    if cohens_d < 0.2:
        effect_size = "Negligible"
    elif cohens_d < 0.5:
        effect_size = "Small"
    elif cohens_d < 0.8:
        effect_size = "Medium"
    else:
        effect_size = "Large"
    
    print(f"  Effect Size: {effect_size}")
    
    if test_result['p_value'] < 0.05:
        print(f"  **RESULT: Statistically significant difference (p < 0.05)**")
        if test_result['diff'] < 0:
            print(f"  **CONCLUSION: T-CCD shows significantly lower {error_name.lower()}**")
        else:
            print(f"  **CONCLUSION: Swept AABB shows significantly lower {error_name.lower()}**")
    else:
        print(f"  **RESULT: No statistically significant difference (p ≥ 0.05)**")
        print(f"  **CONCLUSION: Both algorithms show equivalent {error_name.lower()} performance**")

CONSERVATION ERROR STATISTICAL ANALYSIS

**Table 3: Conservation Error Comparison Results**
--------------------------------------------------------------------------------


,Error Type,T-CCD Mean ± SD,Swept AABB Mean ± SD,Mean Difference,P-value,Statistical Significance,Cohen's d
0,Energy Error,0.0069 ± 0.0054,0.0069 ± 0.0055,0.0000,8.3011e-01,NO,0.009
1,X-axis Error,1.3578 ± 8.1239,2.0314 ± 20.7423,-0.6736,2.9503e-01,NO,-0.043
2,Y-axis Error,1.2876 ± 6.4635,1.6353 ± 12.7127,-0.3477,3.9842e-01,NO,-0.034



DETAILED STATISTICAL RESULTS

**Energy Conservation Error:**
--------------------------------------------------
  T-CCD: 0.0069 ± 0.0054
  Swept AABB: 0.0069 ± 0.0055
  Mean Difference: 0.0000
  95% Confidence Interval: [-0.0004, 0.0005]
  T-statistic: 0.215
  P-value: 8.3011e-01
  Cohen's d: 0.009
  Effect Size: Negligible
  **RESULT: No statistically significant difference (p ≥ 0.05)**
  **CONCLUSION: Both algorithms show equivalent energy conservation error performance**

**X-axis Positional Error:**
--------------------------------------------------
  T-CCD: 1.3578 ± 8.1239
  Swept AABB: 2.0314 ± 20.7423
  Mean Difference: -0.6736
  95% Confidence Interval: [-1.9350, 0.5878]
  T-statistic: -1.047
  P-value: 2.9503e-01
  Cohen's d: -0.043
  Effect Size: Negligible
  **RESULT: No statistically significant difference (p ≥ 0.05)**
  **CONCLUSION: Both algorithms show equivalent x-axis positional error performance**

**Y-axis Positional Error:**
----------------------------------------